# UK Car Emissions Analysis

This notebook analyzes UK car emissions and ZEV mandate scenarios using data from various government sources.

## Setup Instructions

Before running this notebook:

1. Run the data downloader script:
   ```bash
   python download_data.py
   ```
   This will download all required datasets from government sources.

2. If any datasets require manual download (like emissions data), the script will provide instructions.

3. Once all data is downloaded, you can proceed with the analysis below.

## Required Data Files

The following files should be present in the `data/` directory:
- road-transport-fuel-consumption-tables.ods
- veh0105.ods
- veh0142.ods
- traffic_data.csv
- 2306200930_DA_GHGI_1990-2021_Final_v3.1.xlsx (may require manual download)

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import plotly.graph_objects as go
from datetime import datetime
import pickle
from pathlib import Path
from google.cloud import bigquery

# Check required data files exist
data_dir = Path('data')
required_files = [
    'road-transport-fuel-consumption-tables.ods',
    'veh0105.ods',
    'veh0142.ods',
    'traffic_data.csv',
    '2306200930_DA_GHGI_1990-2021_Final_v3.1.xlsx'
]

missing_files = [f for f in required_files if not (data_dir / f).exists()]
if missing_files:
    print('Missing required data files:')
    print('\n'.join(f'- {f}' for f in missing_files))
    print('\nPlease run download_data.py to fetch the required datasets.')
    raise FileNotFoundError('Required data files missing')

# Load plotting parameters
with open('design.pkl','rb') as f:
    set_params = pickle.load(f)
plt.rcParams.update(set_params)
plt.rc('font', family='sans-serif')

# Constants
LAD_CODE = 'K02000001'  # UK
GEOGRAPHY_NAME = 'United Kingdom'
BODY_TYPE = 'Cars'

## 1. Data Loading and Preprocessing

First, we'll load and preprocess the downloaded data.

In [ ]:
def load_fuel_consumption_data():
    """Load and process fuel consumption data"""
    years = list(range(2005, 2022))
    
    frames = []
    for year in years:
        df = pd.read_excel('data/road-transport-fuel-consumption-tables.ods', 
                          header=3, sheet_name=str(year))
        df = df[df['Local Authority Code'] == LAD_CODE]
        df['year'] = year
        frames.append(df)

    fc = pd.concat(frames)
    fc.set_index('year', inplace=True)
    fc.index = pd.to_datetime(fc.index, format='%Y')
    return fc

def clean_fuel_type(x):
    """Standardize fuel type names"""
    replacements = {
        'Battery electric': 'Pure Electric',
        'Other fuels': 'Other',
        'Other fuel types': 'Other',
        'Plug-in hybrid electric (diesel)': 'Diesel',
        'Range extended electric': 'Pure Electric',
        'Plug-in hybrid electric (petrol)': 'Hybrid',
        'Hybrid electric (petrol)': 'Hybrid',
        'Hybrid electric (diesel)': 'Diesel',
        'Fuel cell electric': 'Pure Electric',
        'Gas': 'Other'
    }
    return replacements.get(x, x)

def load_vehicle_statistics():
    """Load and process vehicle licensing statistics"""
    # Process VEH0105
    veh0105 = pd.read_excel('data/veh0105.ods', sheet_name='VEH0105', header=4)
    veh0105 = veh0105[
        (veh0105.BodyType != 'Total') &
        (veh0105['Fuel [note 2]'] != 'Total') &
        (veh0105['Keepership [note 3]'] != 'Total') &
        (veh0105['ONS Code [note 6]'] == LAD_CODE)
    ]
    
    # Process VEH0142
    veh0142 = pd.read_excel('data/veh0142.ods', sheet_name='VEH0142', header=4)
    veh0142 = veh0142[
        (veh0142['Fuel'] != 'Total') &
        (veh0142['BodyType'] != 'Total') &
        (veh0142['Keepership [note 3]'] != 'Total') &
        (veh0142['ONS Code [note 6]'] == LAD_CODE)
    ]
    
    # Combine and process
    for df in [veh0105, veh0142]:
        df.rename(columns={
            'Fuel [note 2]': 'Fuel',
            'Keepership [note 3]': 'Keepership'
        }, inplace=True)
        
    pc = pd.concat([veh0142, veh0105])
    pc['Fuel'] = pc.Fuel.apply(clean_fuel_type)
    return pc

def load_traffic_data():
    """Load traffic data"""
    return pd.read_csv('data/traffic_data.csv')

def load_emissions_data():
    """Load emissions data"""
    em = pd.read_excel('data/2306200930_DA_GHGI_1990-2021_Final_v3.1.xlsx', 
                       sheet_name='UK By Source', header=15, usecols='B:AD')
    em['NCFormat'] = em.NCFormat.fillna(method='ffill')
    return em

## 2. Data Analysis

Now we'll analyze the data to understand current trends and project future scenarios.

In [ ]:
def calculate_vehicle_miles_traveled(cars_df, mileage_df):
    """Calculate vehicle miles traveled by fuel type"""
    df = cars_df.merge(mileage_df, how='left', left_index=True, right_index=True)
    
    df['diesel_vmt'] = df.Diesel * df.diesel_mileage
    df['petrol_vmt'] = df.Petrol * df.petrol_mileage
    df['electric_vmt'] = df['Pure Electric'] * df.electric_mileage
    df['hybrid_vmt'] = df.Hybrid * df.hybrid_mileage
    
    return df

def calculate_emissions(df):
    """Calculate emissions from fuel consumption"""
    # Convert to proper units and apply emission factors
    df['diesel_emissions'] = df.car_diesel_consumption.apply(
        lambda x: ((1000*x)*11629.9998357937)*.24115)
    df['petrol_emissions'] = df.car_petrol_consumption.apply(
        lambda x: ((1000*x)*11629.9998357937)*.22719)
    return df

## 3. ZEV Mandate Scenarios

Here we'll model different scenarios based on the ZEV mandate targets.

In [ ]:
def model_zev_scenario(start_year=2024, end_year=2050, 
                      zev_mandate_targets=None,
                      market_growth=1.0):
    """Model vehicle fleet evolution under ZEV mandate scenario"""
    if zev_mandate_targets is None:
        zev_mandate_targets = [
            .181, .24, .30, .38, .52, .66, .80, .84, .88, .92, .96, 1.0
        ]
    
    years = pd.date_range(start=f"{start_year}-01-01", 
                         end=f"{end_year}-01-01", 
                         freq='YS')
    
    # Initialize projection dataframe
    dfp = pd.DataFrame(index=years)
    
    # Project sales
    sales = 1900000  # baseline annual sales
    total_annual_sales = []
    
    for year in years:
        sales *= market_growth
        total_annual_sales.append(sales)
    
    # Calculate ZEV trajectory
    zev_trajectory = []
    for i, year in enumerate(years):
        if i < len(zev_mandate_targets):
            zev_trajectory.append(zev_mandate_targets[i])
        else:
            zev_trajectory.append(1.0)
    
    # Calculate sales by powertrain
    dfp['new_electric'] = [z * s for z, s in zip(zev_trajectory, total_annual_sales)]
    ice_sales = [s - e for s, e in zip(total_annual_sales, dfp.new_electric)]
    
    diesel_share = 0.11  # current diesel share of ICE sales
    hybrid_share = 0.4   # hybrid share of remaining ICE
    
    dfp['new_diesel'] = [s * diesel_share for s in ice_sales]
    dfp['new_hybrid'] = [(s - d) * hybrid_share 
                         for s, d in zip(ice_sales, dfp.new_diesel)]
    dfp['new_petrol'] = [s - d - h 
                         for s, d, h in zip(ice_sales, 
                                           dfp.new_diesel, 
                                           dfp.new_hybrid)]
    
    return dfp

def calculate_fleet_composition(sales_projection, retirement_years=18):
    """Calculate fleet composition based on sales projection"""
    # Initialize with current fleet numbers
    fleet = pd.DataFrame(index=sales_projection.index)
    
    # Calculate retirements and running total for each powertrain
    for powertrain in ['diesel', 'petrol', 'electric', 'hybrid']:
        sales_col = f'new_{powertrain}'
        fleet[powertrain] = sales_projection[sales_col].cumsum()
        
        # Apply retirements after specified years
        retirements = sales_projection[sales_col].shift(retirement_years).fillna(0)
        fleet[powertrain] -= retirements.cumsum()
    
    return fleet

## 4. Visualization Functions

Functions to create various visualizations of the analysis results.

In [ ]:
def plot_fleet_composition(fleet_df):
    """Plot fleet composition over time"""
    fig = go.Figure()
    
    colors = {
        'diesel': 'black',
        'electric': 'green',
        'petrol': 'blue',
        'hybrid': 'light blue'
    }
    
    for powertrain, color in colors.items():
        fig.add_trace(go.Scatter(
            x=fleet_df.index,
            y=fleet_df[powertrain],
            name=powertrain.capitalize(),
            mode='lines',
            line=dict(color=color)
        ))
    
    fig.update_layout(
        title='Projected UK Car Fleet Composition',
        xaxis=dict(title='Year'),
        yaxis=dict(title='Number of Vehicles'),
        legend=dict(x=0, y=1, bgcolor='rgba(255, 255, 255, 0.5)')
    )
    
    return fig

def plot_emissions_projection(historical_df, forecast_df):
    """Plot historical and projected emissions"""
    fig = go.Figure()
    
    # Historical emissions
    historical = historical_df[['diesel_emissions', 'petrol_emissions']].sum(axis=1)/1000000
    
    # Forecast emissions
    forecast = forecast_df[['diesel_emissions', 'petrol_emissions']].sum(axis=1)/1000000
    
    fig.add_trace(go.Scatter(
        x=historical.index,
        y=historical,
        name='Historical Emissions',
        mode='lines',
        line=dict(color='blue')
    ))
    
    fig.add_trace(go.Scatter(
        x=forecast.index,
        y=forecast,
        name='Projected Emissions',
        mode='lines',
        line=dict(color='red')
    ))
    
    fig.update_layout(
        title='UK Car Emissions: Historical and Projected',
        xaxis=dict(title='Year'),
        yaxis=dict(title='Emissions (MtCO2e)'),
        legend=dict(x=0, y=1)
    )
    
    return fig

## 5. Running the Analysis

Execute the analysis and generate visualizations.

In [ ]:
# Load data
fuel_consumption = load_fuel_consumption_data()
vehicle_stats = load_vehicle_statistics()
traffic_data = load_traffic_data()
emissions_data = load_emissions_data()

# Model ZEV mandate scenario
sales_projection = model_zev_scenario()
fleet_projection = calculate_fleet_composition(sales_projection)

# Create visualizations
fleet_plot = plot_fleet_composition(fleet_projection)
fleet_plot.show()

emissions_plot = plot_emissions_projection(historical_df, forecast_df)
emissions_plot.show()